# 🎬 10Eros LTX 2.3 — Image-to-Video en Google Colab (versión GGUF)

Copia independiente del Space `signsur4739379373/LTX-2.3-10Eros`, optimizada para el disco de Colab gratuito:

| Optimización | Ahorro |
|---|---|
| Checkpoint fp8 (29.2 GB) → **GGUF Q4_K_S (13.2 GB)** + VAEs sueltos (1.8 GB) | **~14 GB** |
| LoRAs opcionales reemplazadas por las tuyas de CivitAI (no se descargan las originales) | variable |
| Limpieza de cachés pip/apt/git | ~5-10 GB |

**Footprint estimado: ~75 GB** (antes: >112 GB)

---
**Instrucciones:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`
2. Ejecuta las celdas **en orden** (la 5 de CivitAI es opcional)
3. La celda 7 valida tu GPU/disco y desbloquea los modelos ejecutables en tu entorno
4. La celda 8 abre la interfaz web (enlace `gradio.live`)

## 📊 Celda 1 — Verificar GPU y CUDA

In [ ]:
# ─── Verificación de GPU ───────────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("❌ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU")
print(result.stdout)

# Detectar versión de CUDA del driver
cuda_ver_line = [l for l in result.stdout.splitlines() if 'CUDA Version' in l]
if cuda_ver_line:
    cuda_str = cuda_ver_line[0].split('CUDA Version:')[-1].strip().split()[0]
    cuda_major, cuda_minor = int(cuda_str.split('.')[0]), int(cuda_str.split('.')[1])
    print(f"\n✅ CUDA detectado: {cuda_str}")
    if cuda_major >= 12 and cuda_minor >= 8:
        TORCH_CUDA_TAG = 'cu128'
    elif cuda_major >= 12 and cuda_minor >= 4:
        TORCH_CUDA_TAG = 'cu124'
    elif cuda_major >= 12:
        TORCH_CUDA_TAG = 'cu121'
    else:
        TORCH_CUDA_TAG = 'cu118'
    print(f"📦 Se usará índice PyTorch: {TORCH_CUDA_TAG}")
else:
    TORCH_CUDA_TAG = 'cu121'
    print("⚠️  No se pudo determinar CUDA, usando cu121 como fallback")

os.environ['TORCH_CUDA_TAG'] = TORCH_CUDA_TAG

# Disco disponible
import shutil
du = shutil.disk_usage('/content')
print(f"\n💽 Disco: {(du.total-du.free)/1024**3:.1f} GB usados / {du.total/1024**3:.1f} GB totales ({du.free/1024**3:.1f} GB libres)")
print("✅ GPU lista para usar")

## 🔑 Celda 2 — Tokens (HuggingFace + CivitAI)

> ℹ️ **Aclaración**: el token de HF **no acelera** las descargas — sirve para repos privados/gated y para evitar errores de rate-limit (429). Lo que sí acelera las descargas (hasta 5x) es `hf_transfer`, que esta celda activa automáticamente.
>
> - **HF_TOKEN**: créalo en https://huggingface.co/settings/tokens (tipo `read`)
> - **CIVITAI_TOKEN**: créalo en https://civitai.com/user/account → API Keys (necesario para descargar la mayoría de LoRAs de CivitAI)

In [ ]:
# ─── Configuración de tokens ───────────────────────────────────────────────
HF_TOKEN = ""        #@param {type:"string"}
CIVITAI_TOKEN = ""   #@param {type:"string"}

import os, subprocess

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    print("✅ HF_TOKEN configurado (repos privados/gated + sin rate-limit)")
else:
    print("ℹ️  Sin HF_TOKEN — las descargas públicas funcionarán igual")

if CIVITAI_TOKEN.strip():
    os.environ['CIVITAI_TOKEN'] = CIVITAI_TOKEN.strip()
    print("✅ CIVITAI_TOKEN configurado")
else:
    print("ℹ️  Sin CIVITAI_TOKEN — muchas descargas de CivitAI fallarán con 401")

# hf_transfer: descarga paralela en Rust — esto SÍ acelera las descargas de HF
subprocess.run(['pip', 'install', '-q', 'hf_transfer'], check=False)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print("🚀 hf_transfer activado — descargas de HuggingFace aceleradas")

## ⬇️ Celda 3 — Clonar Space e Instalar Dependencias
> ⚠️ Tarda ~5-10 minutos la primera vez. Al final purga cachés de pip/apt para ahorrar disco.

In [ ]:
# ─── Clonar el Space e instalar dependencias ──────────────────────────────
import os, subprocess, sys

SPACE_REPO = 'https://huggingface.co/spaces/signsur4739379373/LTX-2.3-10Eros'
SPACE_DIR  = '/content/LTX-10Eros'
TORCH_CUDA_TAG = os.environ.get('TORCH_CUDA_TAG', 'cu121')

def run(cmd, **kw):
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, **kw)
    return proc.returncode

run('apt-get install -qq git-lfs > /dev/null && git lfs install --skip-repo')

# Clonar solo el código (sin archivos LFS grandes)
if not os.path.exists(SPACE_DIR):
    print("\n📥 Clonando Space (solo código)...")
    run(f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {SPACE_REPO} {SPACE_DIR}')
elif os.path.exists(f'{SPACE_DIR}/.git'):
    print("✅ Space ya clonado, actualizando...")
    run(f'cd {SPACE_DIR} && GIT_LFS_SKIP_SMUDGE=1 git pull')
else:
    print("✅ Space ya presente (sin .git, no se actualiza)")

print("\n📦 Instalando PyTorch 2.8.0...")
torch_index = f'https://download.pytorch.org/whl/{TORCH_CUDA_TAG}'
rc = run(f'pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --extra-index-url {torch_index}')
if rc != 0:
    print("⚠️  torch 2.8.0 no disponible, instalando 2.4.0 como fallback...")
    run('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --extra-index-url https://download.pytorch.org/whl/cu121')

print("\n📦 Instalando resto de dependencias...")
deps = [
    'torchsde', 'gradio>=5,<6', 'huggingface_hub>=0.34.0', 'transformers>=4.48.0',
    'accelerate>=0.26.0', 'safetensors', 'einops', 'scipy', 'numpy', 'pillow',
    'opencv-python-headless', 'mediapipe', 'av', 'kornia<0.8.0', 'psutil',
    'websocket-client', 'gguf', 'sentencepiece', 'protobuf', 'spandrel',
    'soundfile', 'imageio-ffmpeg', 'rotary_embedding_torch', 'hf_transfer',
]
run('pip install -q ' + ' '.join(f'"{d}"' for d in deps))

# ─── Limpieza de cachés (ahorro de disco) ──────────────────────────────────
print("\n🧹 Purgando cachés de pip y apt...")
run('pip cache purge')
run('apt-get clean')
run('rm -rf /root/.cache/pip')

import shutil
du = shutil.disk_usage('/content')
print(f"\n💽 Disco libre: {du.free/1024**3:.1f} GB")
print("✅ Dependencias instaladas y cachés purgadas")

## 🩹 Celda 4 — Parchear módulo `spaces` (ZeroGPU → Colab)

In [ ]:
# ─── Mock del módulo 'spaces' de HuggingFace ZeroGPU ─────────────────────
# En Colab la GPU siempre está disponible, no hace falta el decorator @spaces.GPU
import sys, types, os

mock_spaces = types.ModuleType('spaces')

def _gpu_decorator(*args, **kwargs):
    if len(args) == 1 and callable(args[0]) and not kwargs:
        return args[0]
    def decorator(fn):
        return fn
    return decorator

mock_spaces.GPU = _gpu_decorator
mock_spaces.ZeroGPU = _gpu_decorator
sys.modules['spaces'] = mock_spaces

SPACE_DIR = '/content/LTX-10Eros'
if SPACE_DIR not in sys.path:
    sys.path.insert(0, SPACE_DIR)
os.chdir(SPACE_DIR)
print("✅ Módulo 'spaces' parcheado — GPU de Colab se usará directamente")

## 🎨 Celda 5 — LoRAs personalizadas de CivitAI (opcional)

Hay **6 slots intercambiables** en la UI (LoRAs que solo usa el preset "experimental #1", o ninguno):

| Slot | LoRA original | ¿Algún preset la usa? |
|---|---|---|
| `slot1` | Cinematic Hardcut | No (0.0 en todos) |
| `slot2` | Synth | Solo experimental #1 |
| `slot3` | Plora Sulfer | Solo experimental #1 |
| `slot4` | OmniNFT RL bf16 (la más pesada) | Solo experimental #1 |
| `slot5` | Better Motion | Solo experimental #1 |
| `slot6` | Physics V2 | Solo experimental #1 |

- Pega la **URL de CivitAI** (ej: `https://civitai.com/models/123456?modelVersionId=789012`) o solo el **ID de versión** en el slot que quieras reemplazar.
- Slot vacío (`""`) = se descarga y usa la LoRA original (no pierdes nada).
- Slot reemplazado = **no** se descarga la original (ahorras su tamaño) y el slider de ese slot en la UI controla **tu** LoRA.
- Al ejecutar la celda se consultan los **metadatos de CivitAI** (nombre, base model, trigger words, tamaño) y en la Celda 8 el slider del slot se **renombra** a `custom: <nombre de tu LoRA>`.
- ⚠️ Usa LoRAs entrenadas para **LTX 2.3** — las de SDXL/Flux/Wan no son compatibles.
- Para activar tu LoRA en la UI, sube el slider de su slot (la mayoría arranca en 0).

In [ ]:
# ─── Descarga de LoRAs custom desde CivitAI ───────────────────────────────
CUSTOM_LORAS = {
    # slot   : URL de CivitAI o ID de versión ("" = mantener la original)
    "slot1": "",   # original: Cinematic Hardcut
    "slot2": "",   # original: Synth
    "slot3": "",   # original: Plora Sulfer
    "slot4": "",   # original: OmniNFT RL bf16
    "slot5": "",   # original: Better Motion
    "slot6": "",   # original: Physics V2
}

import os, re, json, pathlib, requests

CIVITAI_TOKEN = os.environ.get('CIVITAI_TOKEN', '')
STAGING = pathlib.Path('/content/custom_loras')
STAGING.mkdir(exist_ok=True)

def resolve_version_id(ref: str):
    """Acepta URL completa de CivitAI o un ID numérico de versión."""
    ref = ref.strip()
    if not ref:
        return None
    if re.fullmatch(r'\d+', ref):
        return ref
    m = re.search(r'modelVersionId=(\d+)', ref)
    if m:
        return m.group(1)
    m = re.search(r'civitai\.com/models/(\d+)', ref)
    if m:
        # Sin modelVersionId: consultar la API para tomar la versión más reciente
        r = requests.get(f'https://civitai.com/api/v1/models/{m.group(1)}', timeout=30)
        r.raise_for_status()
        return str(r.json()['modelVersions'][0]['id'])
    raise ValueError(f'Referencia no reconocida: {ref}')

def fetch_version_info(version_id: str) -> dict:
    """Metadatos públicos de la versión: nombre, base model, trigger words, tamaño."""
    try:
        r = requests.get(f'https://civitai.com/api/v1/model-versions/{version_id}',
                         timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        d = r.json()
        f0 = (d.get('files') or [{}])[0]
        return {
            'version_id':    str(d.get('id', version_id)),
            'name':          (d.get('model') or {}).get('name', ''),
            'version':       d.get('name', ''),
            'base_model':    d.get('baseModel', ''),
            'trained_words': d.get('trainedWords') or [],
            'size_mb':       round((f0.get('sizeKB') or 0) / 1024, 1),
            'url':           f"https://civitai.com/models/{d.get('modelId')}?modelVersionId={d.get('id')}",
        }
    except Exception as e:
        print(f'  ⚠️  Sin metadatos de CivitAI ({e}) — se continúa solo con el archivo')
        return {}

def print_lora_info(info: dict):
    if not info:
        return
    print(f'  📄 "{info["name"]}" — versión: {info["version"]}')
    print(f'     base model: {info["base_model"] or "?"} | tamaño: {info["size_mb"]:.0f} MB')
    if info['trained_words']:
        print(f'     trigger words: {", ".join(info["trained_words"][:8])}')
    if info['base_model'] and 'ltx' not in info['base_model'].lower():
        print(f'     ⚠️  base model "{info["base_model"]}" no parece LTX — probablemente incompatible')

def download_civitai(version_id: str, slot: str) -> str:
    url = f'https://civitai.com/api/download/models/{version_id}'
    params = {'token': CIVITAI_TOKEN} if CIVITAI_TOKEN else {}
    with requests.get(url, params=params, stream=True, timeout=60,
                      headers={'User-Agent': 'Mozilla/5.0'}) as r:
        if r.status_code == 401:
            raise RuntimeError(
                f'401 en slot {slot}: CivitAI requiere token para esta descarga. '
                'Configura CIVITAI_TOKEN en la Celda 2.')
        r.raise_for_status()
        cd = r.headers.get('Content-Disposition', '')
        m = re.search(r'filename="?([^";]+)', cd)
        fname = m.group(1) if m else f'civitai_{version_id}.safetensors'
        # sanitizar nombre
        fname = re.sub(r'[^\w.\- ]', '_', fname)
        if not fname.endswith('.safetensors'):
            fname += '.safetensors'
        dest = STAGING / fname
        if dest.exists():
            print(f'  ✅ {fname} ya descargado')
            return fname
        total = int(r.headers.get('Content-Length', 0))
        done = 0
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f'\r  ⬇️  {fname}: {done/1024**2:.0f}/{total/1024**2:.0f} MB', end='')
        print()
        return fname

manifest = {}
for slot, ref in CUSTOM_LORAS.items():
    try:
        vid = resolve_version_id(ref)
    except Exception as e:
        print(f'❌ slot {slot}: {e}')
        continue
    if vid is None:
        continue
    print(f'🎨 Slot "{slot}" ← versión CivitAI {vid}')
    info = fetch_version_info(vid)
    print_lora_info(info)
    try:
        fname = download_civitai(vid, slot)
    except Exception as e:
        print(f'❌ slot {slot}: {e}')
        continue
    manifest[slot] = {'file': fname, **info}

with open(STAGING / 'mapping.json', 'w') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

if manifest:
    print('\n📋 Resumen de slots reemplazados:')
    for slot, entry in manifest.items():
        display = entry.get('name') or pathlib.Path(entry['file']).stem
        print(f'   • slot "{slot}" → {entry["file"]}')
        print(f'     su slider en la UI pasará a llamarse: "custom: {display}"')
    print('\n⚠️  Recuerda subir el slider del slot en la UI para activar tu LoRA')
else:
    print('ℹ️  Sin LoRAs custom — se usarán todas las originales')

## 📊 Celda 6 — Monitor de VRAM, Disco y Tiempo de Sesión
> Ejecuta esta celda **antes** de lanzar la app. Se actualiza cada 5 segundos.

In [ ]:
# ─── Monitor de VRAM + Disco + Tiempo de sesión ───────────────────────────
import torch, time, threading, datetime, shutil
from IPython.display import display, HTML
import ipywidgets as widgets

SESSION_START   = time.time()
COLAB_GPU_LIMIT = 5 * 3600   # ~5 horas (límite aproximado de cuenta gratuita)
_monitor_running = True

def fmt_time(seconds):
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def color_for_pct(pct):
    return '#4CAF50' if pct < 60 else ('#FF9800' if pct < 80 else '#f44336')

def bar(pct, color):
    return (f'<div style="background:#333;border-radius:4px;height:12px;margin-bottom:10px">'
            f'<div style="background:{color};width:{min(pct,100):.1f}%;height:12px;'
            f'border-radius:4px;transition:width .4s"></div></div>')

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
out = widgets.Output()
display(out)

def update_monitor():
    while _monitor_running:
        elapsed   = time.time() - SESSION_START
        remaining = max(0, COLAB_GPU_LIMIT - elapsed)
        t_pct     = min(100, (elapsed / COLAB_GPU_LIMIT) * 100)

        if torch.cuda.is_available():
            v_used  = torch.cuda.memory_allocated() / 1024**2
            v_resv  = torch.cuda.memory_reserved() / 1024**2
            v_total = torch.cuda.get_device_properties(0).total_memory / 1024**2
            v_pct   = (v_used / v_total) * 100 if v_total else 0
        else:
            v_used = v_resv = v_total = v_pct = 0

        du = shutil.disk_usage('/content')
        d_used  = (du.total - du.free) / 1024**3
        d_total = du.total / 1024**3
        d_pct   = (d_used / d_total) * 100 if d_total else 0

        vc, tc, dc = color_for_pct(v_pct), color_for_pct(t_pct), color_for_pct(d_pct)
        now_str = datetime.datetime.now().strftime('%H:%M:%S')

        html = f"""
<div style="font-family:monospace;background:#1e1e1e;color:#e0e0e0;padding:14px 18px;
            border-radius:10px;max-width:580px;border:1px solid #444;margin:4px 0">
  <div style="font-size:15px;font-weight:bold;margin-bottom:10px;color:#90CAF9">
    📊 Monitor — {now_str}</div>
  <div style="margin-bottom:8px">🖥️ <b>GPU:</b> {gpu_name}</div>
  <div style="margin-bottom:4px">💾 <b>VRAM:</b> {v_used:.0f} / {v_total:.0f} MB
    <span style="color:#aaa">(reservada {v_resv:.0f} MB)</span>
    <b style="color:{vc}">{v_pct:.1f}%</b></div>
  {bar(v_pct, vc)}
  <div style="margin-bottom:4px">💽 <b>Disco:</b> {d_used:.1f} / {d_total:.1f} GB
    <b style="color:{dc}">{d_pct:.1f}%</b></div>
  {bar(d_pct, dc)}
  <div style="margin-bottom:4px">⏱️ <b>Sesión:</b> {fmt_time(elapsed)}
    <b style="color:{tc}">{t_pct:.1f}% del límite</b></div>
  {bar(t_pct, tc)}
  <div>⏳ <b>Tiempo restante (aprox):</b>
    <b style="color:{tc}">{fmt_time(remaining)}</b>
    <span style="color:#aaa">(límite ~5h cuenta gratuita)</span></div>
</div>"""
        with out:
            out.clear_output(wait=True)
            display(HTML(html))
        time.sleep(5)

threading.Thread(target=update_monitor, daemon=True).start()
print("✅ Monitor activado (VRAM + disco + tiempo, cada 5 s)")

## 🧮 Celda 7 — Validar entorno y elegir modelo

Detecta tu GPU (VRAM) y el disco libre, y **desbloquea solo las variantes del checkpoint que pueden ejecutarse** en este runtime:

| Entorno | Variantes desbloqueadas (aprox.) |
|---|---|
| Colab gratuito (T4, ~15 GB VRAM) | GGUF hasta `Q4_K_S` (el por defecto) |
| Colab Pro (L4, ~22.5 GB VRAM) | GGUF hasta `Q6_K` |
| A100 (40 GB VRAM) | Todo, incluido el **fp8 original sin cuantizar** (29.2 GB) |

Elige en el desplegable; la Celda 8 usará tu selección automáticamente. Si te saltas esta celda, la Celda 8 usa su parámetro `GGUF_QUANT` como hasta ahora.

In [ ]:
# ─── Validar entorno (GPU / VRAM / disco) y elegir modelo ──────────────────
import os, shutil, subprocess

# Catálogo: (clave, tamaño del checkpoint GB, VRAM mínima estimada GB)
# Los requisitos son estimaciones: tamaño del archivo + margen para
# activaciones/VAEs. FP8_ORIGINAL = checkpoint sin cuantizar (sin parches GGUF).
MODEL_CATALOG = [
    ('Q3_K_S',       10.3, 11.5),
    ('Q3_K_M',       11.2, 12.5),
    ('Q4_0',         12.6, 14.0),
    ('Q4_K_S',       13.2, 14.5),
    ('Q4_K_M',       14.0, 15.5),
    ('Q5_K_S',       15.6, 17.0),
    ('Q5_K_M',       16.1, 17.5),
    ('Q6_K',         18.5, 20.0),
    ('Q8_0',         23.9, 25.5),
    ('FP8_ORIGINAL', 29.2, 31.0),
]
BASE_DISK_GB = 62  # deps + ComfyUI + text encoder + LoRAs + VAEs (estimado)

def _model_label(key, size_gb):
    if key == 'FP8_ORIGINAL':
        return f'fp8 mixed original ({size_gb} GB, máxima calidad)'
    return f'GGUF {key} ({size_gb} GB)'

# Detectar GPU y VRAM total
try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
    gpu_name, vram_mb = (s.strip() for s in out.split(','))
    vram_gb = float(vram_mb) / 1024
except Exception:
    gpu_name, vram_gb = 'no detectada', 0.0

disk_free_gb = shutil.disk_usage('/content').free / 1024**3
print(f'🖥️  GPU: {gpu_name} — {vram_gb:.1f} GB VRAM')
print(f'💽 Disco libre: {disk_free_gb:.1f} GB')
print(f'   (el chequeo de disco asume primera descarga completa: ~{BASE_DISK_GB} GB base + checkpoint;')
print(f'    si ya descargaste los modelos en esta sesión, ignora los candados de disco)\n')

unlocked, rows = [], []
for key, size_gb, vram_req in MODEL_CATALOG:
    need_disk = BASE_DISK_GB + size_gb
    ok_vram = vram_gb >= vram_req
    ok_disk = disk_free_gb >= need_disk
    if ok_vram and ok_disk:
        unlocked.append(key)
        status = '✅ disponible'
    else:
        reasons = []
        if not ok_vram:
            reasons.append(f'VRAM: necesita ~{vram_req:.0f} GB')
        if not ok_disk:
            reasons.append(f'disco: necesita ~{need_disk:.0f} GB')
        status = '🔒 ' + ' | '.join(reasons)
    rows.append((_model_label(key, size_gb), status))

w = max(len(r[0]) for r in rows)
for label, status in rows:
    print(f'  {label:<{w}}  {status}')

if not unlocked:
    raise RuntimeError('❌ Ninguna variante cabe en este entorno. ¿Runtime sin GPU? '
                       'Ve a Entorno de ejecución → Cambiar tipo → GPU')

# Por defecto: Q4_K_S si está desbloqueado (el probado en T4); si no, el mejor disponible
default = 'Q4_K_S' if 'Q4_K_S' in unlocked else unlocked[-1]
os.environ['LTX_MODEL_CHOICE'] = default

import ipywidgets as widgets
from IPython.display import display

_sizes = {k: s for k, s, _ in MODEL_CATALOG}
dd = widgets.Dropdown(
    options=[(_model_label(k, _sizes[k]), k) for k in unlocked],
    value=default,
    description='Modelo:',
    layout=widgets.Layout(width='420px'),
)
def _on_select(change):
    os.environ['LTX_MODEL_CHOICE'] = change['new']
    print(f'✅ Modelo seleccionado: {change["new"]}')
dd.observe(_on_select, names='value')
display(dd)
print(f'\n✅ Selección actual: {default} — cámbiala en el desplegable si quieres; '
      'la Celda 8 la usará automáticamente')

## 🚀 Celda 8 — Parchear y Lanzar la App

Aplica los parches en memoria (el `app.py` clonado no se modifica) y lanza Gradio:
- Usa el **modelo elegido en la Celda 7** (o el parámetro `GGUF_QUANT` de abajo si te la saltaste). Con `FP8_ORIGINAL` se omiten los parches GGUF y se usa el checkpoint original de 29.2 GB.
- Text encoder fp8 (13.2 GB) o fp4 (9.4 GB) si activas `USE_FP4_TEXT_ENCODER`
- Swap de slots por tus LoRAs de CivitAI (Celda 5) + renombrado de sus sliders con el nombre real de la LoRA

> La primera ejecución descarga los modelos (~45-85 GB según la variante, 10-25 min con `hf_transfer`).
> Cuando veas `Running on public URL: https://...gradio.live` → abre ese enlace.

In [ ]:
# ─── Parches GGUF + LoRAs custom + lanzamiento ─────────────────────────────
# Si ejecutaste la Celda 7 (selector de modelo), se usa lo elegido ahí.
# Si te la saltaste, se usa este parámetro como fallback:
GGUF_QUANT = "Q4_K_S"  #@param ["Q3_K_S", "Q3_K_M", "Q4_0", "Q4_K_S", "Q4_K_M", "Q5_K_S", "Q5_K_M", "Q6_K", "Q8_0", "FP8_ORIGINAL"]
USE_FP4_TEXT_ENCODER = False  #@param {type:"boolean"}

import os, sys, json, pathlib
import gradio as gr

MODEL_CHOICE = os.environ.get('LTX_MODEL_CHOICE', GGUF_QUANT)
USE_GGUF = MODEL_CHOICE != 'FP8_ORIGINAL'
if USE_GGUF:
    GGUF_QUANT = MODEL_CHOICE
print(f'🧩 Modelo elegido: {MODEL_CHOICE}'
      + ('' if 'LTX_MODEL_CHOICE' in os.environ else ' (fallback del parámetro GGUF_QUANT — ejecuta la Celda 7 para validar tu entorno)'))

SPACE_DIR = '/content/LTX-10Eros'
os.chdir(SPACE_DIR)
GGUF_FILE = f'10Eros_v1-{GGUF_QUANT}.gguf'

app_py = os.path.join(SPACE_DIR, 'app.py')
with open(app_py, encoding='utf-8') as f:
    source = f.read()

_failed = []
def patch(old, new, label):
    global source
    if old not in source:
        print(f"⚠️  parche '{label}' NO aplicado (¿cambió el código del Space?)")
        _failed.append(label)
        return
    source = source.replace(old, new)
    print(f"✅ parche '{label}' aplicado")

def gpatch(old, new, label):
    """patch() solo en modo GGUF; con FP8_ORIGINAL el checkpoint original se usa tal cual."""
    if USE_GGUF:
        patch(old, new, label)
    else:
        print(f"⏭️  parche '{label}' omitido (fp8 original)")

# ── P1: checkpoint fp8 → GGUF + VAEs + text projection en DOWNLOADS ──
gpatch('''    {
        "repo": "TenStrip/LTX2.3-10Eros",
        "file": "10Eros_v1-fp8mixed_learned.safetensors",
        "dest": MODELS / "checkpoints" / "10Eros_v1-fp8mixed_learned.safetensors",
        "label": "main checkpoint",
    },''',
f'''    {{
        "repo": "vantagewithai/LTX2.3-10Eros-GGUF",
        "file": "{GGUF_FILE}",
        "dest": MODELS / "diffusion_models" / "{GGUF_FILE}",
        "label": "main checkpoint (GGUF {GGUF_QUANT})",
    }},
    {{
        "repo": "Kijai/LTX2.3_comfy",
        "file": "vae/LTX23_video_vae_bf16.safetensors",
        "dest": MODELS / "vae" / "LTX23_video_vae_bf16_KJ.safetensors",
        "label": "video VAE (Kijai)",
    }},
    {{
        "repo": "Kijai/LTX2.3_comfy",
        "file": "vae/LTX23_audio_vae_bf16.safetensors",
        "dest": MODELS / "vae" / "LTX23_audio_vae_bf16_KJ.safetensors",
        "label": "audio VAE (Kijai)",
    }},
    {{
        "repo": "Kijai/LTX2.3_comfy",
        "file": "text_encoders/ltx-2.3_text_projection_bf16.safetensors",
        "dest": MODELS / "text_encoders" / "ltx-2.3_text_projection_bf16.safetensors",
        "label": "text projection (Kijai)",
    }},''',
'P1: DOWNLOADS checkpoint→GGUF + VAEs')

# ── P2: workflow MSR (runexx) — nodo 59 UNETLoader → UnetLoaderGGUF ──
gpatch('''            node["type"] = "CheckpointLoaderSimple"
            node["widgets_values"] = ["10Eros_v1-fp8mixed_learned.safetensors"]''',
f'''            node["type"] = "UnetLoaderGGUF"
            node["widgets_values"] = ["{GGUF_FILE}"]''',
'P2: runexx nodo 59 → UnetLoaderGGUF')

# ── P3: runexx nodo 57 — restaurar DualCLIPLoader original (gemma + text projection) ──
gpatch('''        elif nid == RUNEXX_NODE_CLIP_LOADER:
            node["type"] = "LTXAVTextEncoderLoader"
            node["widgets_values"] = [
                "gemma_3_12B_it_fp8_scaled.safetensors",
                "10Eros_v1-fp8mixed_learned.safetensors",
                "default",
            ]''',
'''        elif nid == RUNEXX_NODE_CLIP_LOADER:
            node["type"] = "DualCLIPLoader"
            node["widgets_values"] = [
                "gemma_3_12B_it_fp8_scaled.safetensors",
                "ltx-2.3_text_projection_bf16.safetensors",
                "ltxv",
                "default",
            ]''',
'P3: runexx nodo 57 → DualCLIPLoader')

# ── P4: runexx nodo 53 — restaurar VAELoaderKJ original (audio VAE) ──
gpatch('''        elif nid == RUNEXX_NODE_VAE_AUDIO:
            node["type"] = "LTXVAudioVAELoader"
            node["widgets_values"] = ["10Eros_v1-fp8mixed_learned.safetensors"]''',
'''        elif nid == RUNEXX_NODE_VAE_AUDIO:
            node["type"] = "VAELoaderKJ"
            node["widgets_values"] = ["LTX23_audio_vae_bf16_KJ.safetensors", "main_device", "bf16"]''',
'P4: runexx nodo 53 → VAELoaderKJ')

# ── P5: runexx — mantener el VAELoader de video original (nodo 56) ──
gpatch('''    skip_ids = {
        RUNEXX_NODE_VAE_VIDEO,
        RUNEXX_NODE_VAE_TINY,''',
'''    skip_ids = {
        RUNEXX_NODE_VAE_TINY,''',
'P5: runexx no saltar nodo 56 (video VAE)')

gpatch('''        (RUNEXX_NODE_VAE_VIDEO, 0): [str(RUNEXX_NODE_UNET_LOADER), 2],
''', '', 'P6: runexx quitar rewire VAE→checkpoint')

# ── P7: funciones inyectadas (patch del workflow default + instalador de loras custom) ──
_PRELUDE = '''
def _colab_install_custom_loras():
    src_dir = pathlib.Path("/content/custom_loras")
    if not src_dir.exists():
        return
    dest_dir = MODELS / "loras" / "ltx23" / "custom"
    dest_dir.mkdir(parents=True, exist_ok=True)
    for f in src_dir.glob("*.safetensors"):
        dest = dest_dir / f.name
        if not dest.exists():
            try:
                os.link(f, dest)
            except OSError:
                shutil.copy2(f, dest)
            print(f"[colab] custom lora instalada: {f.name}", flush=True)


def _colab_gguf_patch_default(api):
    \'\'\'Reemplaza CheckpointLoaderSimple (fp8 29GB) por UnetLoaderGGUF +
    VAEs sueltos + DualCLIPLoader (config probada del workflow runexx).\'\'\'
    api["646"] = {"class_type": "UnetLoaderGGUF",
                  "inputs": {"unet_name": "__COLAB_GGUF_FILE__"}}
    api["64600"] = {"class_type": "VAELoader",
                    "inputs": {"vae_name": "LTX23_video_vae_bf16_KJ.safetensors"}}
    api["617"] = {"class_type": "VAELoaderKJ",
                  "inputs": {"vae_name": "LTX23_audio_vae_bf16_KJ.safetensors",
                             "device": "main_device", "weight_dtype": "bf16"}}
    api["616"] = {"class_type": "DualCLIPLoader",
                  "inputs": {"clip_name1": "gemma_3_12B_it_fp8_scaled.safetensors",
                             "clip_name2": "ltx-2.3_text_projection_bf16.safetensors",
                             "type": "ltxv", "device": "default"}}
    for _node in api.values():
        _inputs = _node.get("inputs", {})
        for _k, _v in list(_inputs.items()):
            if (isinstance(_v, list) and len(_v) == 2
                    and str(_v[0]) == "646" and str(_v[1]) == "2"):
                _inputs[_k] = ["64600", 0]
    print("[colab] workflow default parcheado a GGUF", flush=True)


'''
patch('def _ensure_comfy() -> None:',
      _PRELUDE + 'def _ensure_comfy() -> None:',
      'P7: inyectar funciones colab')
source = source.replace('__COLAB_GGUF_FILE__', GGUF_FILE)

# ── P8: aplicar patch GGUF al workflow default tras la conversión ──
gpatch('        _workflow_cache["default"] = _convert_workflow(path)',
'''        _workflow_cache["default"] = _convert_workflow(path)
        _colab_gguf_patch_default(_workflow_cache["default"])''',
'P8: patch default workflow')

# ── P9: instalar loras custom al preparar ComfyUI ──
patch('''
    _comfy_ready = True
''',
'''
    _colab_install_custom_loras()
    _comfy_ready = True
''',
'P9: hook instalador loras custom')

# ── P10: swap de slots opcionales por LoRAs de CivitAI + renombrar sliders ──
# Claves = slots genéricos slot1..slot6; los valores son snippets VERBATIM de
# app.py (no cambiarlos salvo que cambie el código del Space).
# slot1=Cinematic Hardcut, slot2=Synth, slot3=Plora Sulfer,
# slot4=OmniNFT RL bf16, slot5=Better Motion, slot6=Physics V2
_CONST_LINES = {
    'slot1': 'HARDCUT_LORA_FILENAME = "ltx23/2508281_LTX-2.3_Cinematic hardcut.safetensors"',
    'slot2': 'SYNTH_LORA_FILENAME = "ltx23/2509189_Synth_01_rank32.safetensors"',
    'slot3': 'PLORA_LORA_FILENAME = "ltx23/2598050_plora_sulfer_v1.2-step00008500.safetensors"',
    'slot4': 'OMNINFT_BF16_LORA_FILENAME = "ltx23/LTX-2.3-OmniNFT-RL-Lora_bf16.safetensors"',
    'slot5': 'BETTER_MOTION_LORA_FILENAME = "ltx23/2344781_Sulphur_LTX 2.3_better_motion.safetensors"',
    'slot6': 'PHYSICS_V2_LORA_FILENAME = "ltx23/2592090_LTX2.3_Physics_V2_000002000.safetensors"',
}
# Etiquetas de los sliders (video, audio) de cada slot en la UI de Gradio
_LABEL_LINES = {
    'slot1': ('label="cinematic hardcut (0 = off)",',        'label="cinematic hardcut (audio)",'),
    'slot2': ('label="synth lora (0 = off)",',               'label="synth (audio)",'),
    'slot3': ('label="plora (0 = off)",',                    'label="plora (audio)",'),
    'slot4': ('label="omninft RL bf16 / kijai (0 = off)",',  'label="omninft RL bf16 / kijai (audio)",'),
    'slot5': ('label="better motion / mistic (0 = off)",',   'label="better motion / mistic (audio)",'),
    'slot6': ('label="physics v2 / mistic (0 = off)",',      'label="physics v2 / mistic (audio)",'),
}
_DL_BLOCKS = {
    'slot1': '''    {
        "repo": "signsur4739379373/archive",
        "file": "2508281_LTX-2.3_Cinematic hardcut.safetensors",
        "dest": MODELS / "loras" / "ltx23" / "2508281_LTX-2.3_Cinematic hardcut.safetensors",
        "label": "cinematic hardcut lora",
    },
''',
    'slot2': '''    {
        "repo": "signsur4739379373/archive",
        "file": "2509189_Synth_01_rank32.safetensors",
        "dest": MODELS / "loras" / "ltx23" / "2509189_Synth_01_rank32.safetensors",
        "label": "synth lora",
    },
''',
    'slot3': '''    {
        "repo": "signsur4739379373/archive",
        "file": "2598050_plora_sulfer_v1.2-step00008500.safetensors",
        "dest": MODELS / "loras" / "ltx23" / "2598050_plora_sulfer_v1.2-step00008500.safetensors",
        "label": "plora",
    },
''',
    'slot4': '''    {
        "repo": "Kijai/LTX2.3_comfy",
        "file": "loras/LTX-2.3-OmniNFT-RL-Lora_bf16.safetensors",
        "dest": MODELS / "loras" / "ltx23" / "LTX-2.3-OmniNFT-RL-Lora_bf16.safetensors",
        "label": "omninft RL bf16 lora",
    },
''',
    'slot5': '''    {
        "repo": "signsur4739379373/archive",
        "file": "2344781_Sulphur_LTX 2.3_better_motion.safetensors",
        "dest": MODELS / "loras" / "ltx23" / "2344781_Sulphur_LTX 2.3_better_motion.safetensors",
        "label": "better motion lora (mistic)",
    },
''',
    'slot6': '''    {
        "repo": "signsur4739379373/archive",
        "file": "2592090_LTX2.3_Physics_V2_000002000.safetensors",
        "dest": MODELS / "loras" / "ltx23" / "2592090_LTX2.3_Physics_V2_000002000.safetensors",
        "label": "physics v2 lora (mistic)",
    },
''',
}
# Compatibilidad con mapping.json generados con los nombres antiguos de slot
_LEGACY_SLOTS = {
    'hardcut': 'slot1', 'synth': 'slot2', 'plora': 'slot3',
    'omninft_bf16': 'slot4', 'better_motion': 'slot5', 'physics_v2': 'slot6',
}

_mapping_file = pathlib.Path('/content/custom_loras/mapping.json')
_lora_mapping = json.loads(_mapping_file.read_text()) if _mapping_file.exists() else {}
for _slot, _entry in _lora_mapping.items():
    _slot = _LEGACY_SLOTS.get(_slot, _slot)
    if isinstance(_entry, str):
        _entry = {'file': _entry}  # compatibilidad con mapping.json antiguo
    _fname = _entry['file']
    _line = _CONST_LINES.get(_slot)
    if not _line:
        continue
    _const = _line.split(' =')[0]
    patch(_line, f'{_const} = "ltx23/custom/{_fname}"', f'P10: slot {_slot} → {_fname}')
    # no descargar la lora original reemplazada
    patch(_DL_BLOCKS[_slot], '', f'P10: skip descarga original de {_slot}')
    # renombrar los sliders del slot con el nombre real de la LoRA custom
    _display = (_entry.get('name') or pathlib.Path(_fname).stem).replace('"', "'").strip()[:40]
    _vid_label, _aud_label = _LABEL_LINES[_slot]
    patch(_vid_label, f'label="custom: {_display} (0 = off)",', f'P10: label video {_slot}')
    patch(_aud_label, f'label="custom: {_display} (audio)",', f'P10: label audio {_slot}')
    if _entry.get('trained_words'):
        print(f'   💬 trigger words de "{_display}": {", ".join(_entry["trained_words"][:8])}')

# ── P11 (opcional): text encoder fp4 (9.4 GB en vez de 13.2 GB) ──
if USE_FP4_TEXT_ENCODER:
    n = source.count('gemma_3_12B_it_fp8_scaled.safetensors')
    source = source.replace('gemma_3_12B_it_fp8_scaled.safetensors',
                            'gemma_3_12B_it_fp4_mixed.safetensors')
    print(f'✅ P11: text encoder → fp4_mixed ({n} referencias)')

if _failed:
    print(f'\n⚠️  {len(_failed)} parches fallaron: {_failed}')
    print('   La app puede intentar descargar el checkpoint fp8 de 29 GB. Revisa antes de continuar.')
else:
    print('\n✅ Todos los parches aplicados correctamente')

# ── Forzar share=True en Gradio ──
_original_launch = gr.Blocks.launch
def _patched_launch(self, *args, **kwargs):
    kwargs.setdefault('share', True)
    kwargs.setdefault('server_port', 7860)
    print('\n🌐 Iniciando servidor Gradio con enlace público...')
    return _original_launch(self, *args, **kwargs)
gr.Blocks.launch = _patched_launch

# ── Ejecutar app.py parcheado (el archivo en disco NO se modifica) ──
_model_desc = f'GGUF {GGUF_QUANT}' if USE_GGUF else 'fp8 mixed original — sin parches GGUF'
print(f'\n▶️  Ejecutando app.py parcheado ({_model_desc})')
print('📦 Primera vez: descarga de modelos ~45-85 GB según el modelo (10-25 min)...\n')
exec(compile(source, app_py, 'exec'), {'__file__': app_py, '__name__': '__main__'})

## 🧹 Celda 9 — Limpieza profunda de disco (opcional)
> Ejecútala **después** de que la app haya arrancado al menos una vez (detén la Celda 8 primero). Borra los `.git` de ComfyUI/nodos y cachés residuales — los modelos no se tocan.

In [ ]:
# ─── Limpieza profunda (no toca modelos) ───────────────────────────────────
import subprocess, shutil, pathlib

before = shutil.disk_usage('/content').free

cmds = [
    'pip cache purge',
    'apt-get clean',
    'rm -rf /root/.cache/pip',
    # .git de ComfyUI y custom nodes (la app no los re-clona si el dir existe)
    'find /content/LTX-10Eros/ComfyUI -name .git -type d -prune -exec rm -rf {} +',
]
for c in cmds:
    print(f'$ {c}')
    subprocess.run(c, shell=True)

after = shutil.disk_usage('/content').free
print(f'\n🧹 Liberados {(after-before)/1024**3:.2f} GB')
print(f'💽 Disco libre: {after/1024**3:.1f} GB')

# Top de carpetas más pesadas para diagnóstico
print('\n📂 Carpetas más pesadas:')
subprocess.run('du -h --max-depth=2 /content/LTX-10Eros 2>/dev/null | sort -rh | head -15', shell=True)

## 🛑 Celda 10 — Detener monitor y liberar GPU (opcional)

In [ ]:
# ─── Detener monitor y liberar memoria GPU ─────────────────────────────────
import torch, gc

try:
    _monitor_running = False
    print("⏹️  Monitor detenido")
except NameError:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    used  = torch.cuda.memory_allocated() / 1024**2
    total = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f"🧹 Caché CUDA limpiada — VRAM: {used:.0f} / {total:.0f} MB")

print("\n✅ Listo. Re-ejecuta la Celda 8 para reiniciar la app.")

---
## ℹ️ Notas importantes

| Tema | Detalle |
|------|--------|
| **Cuantización** | `Q4_K_S` (13.2 GB) cabe en la VRAM de la T4. `Q5_K_S`/`Q6_K` mejoran calidad pero requieren GPU con más VRAM (Colab Pro: L4/A100). La **Celda 7** valida tu entorno y desbloquea solo las variantes ejecutables (incluido el fp8 original en A100) |
| **Text encoder fp4** | Activa `USE_FP4_TEXT_ENCODER` si te quedas corto de disco o RAM (−3.8 GB, leve pérdida de calidad de prompt) |
| **LoRA destilada** | Se mantiene (7.6 GB): es **necesaria** para el sampling de pocos pasos del workflow principal |
| **Slots custom** | El slider se **renombra** a `custom: <nombre>` con los metadatos de CivitAI y controla tu LoRA. Súbelo para activarla |
| **Preset experimental #1** | Si reemplazas slots, ese preset usará tus LoRAs en lugar de las originales |
| **Token HF** | No acelera descargas; sirve para repos gated y rate-limits. `hf_transfer` (Celda 2) sí acelera |
| **Persistencia** | `/content` se borra al reiniciar el runtime — habrá que re-descargar modelos |
| **Independencia** | Todo se descarga de repos públicos de HF (no del Space). Si el Space muere, cambia el clone de la Celda 3 por una copia local de `app.py` + `runexx_msr_workflow.json` |

### 💾 Respaldo recomendado (por si borran el Space)
Guarda una copia de `app.py` y `runexx_msr_workflow.json` en tu Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/LTX-10Eros-backup
!cp /content/LTX-10Eros/app.py /content/LTX-10Eros/runexx_msr_workflow.json /content/drive/MyDrive/LTX-10Eros-backup/
```
Si el Space desaparece, crea `/content/LTX-10Eros/` y copia esos 2 archivos ahí antes de ejecutar la Celda 7.